# cLISI and iLISI of the C2S encodings (Danenberg 2022 and Jackson 2020 IMC)

**Question.** In the pooled profile space (cells from several patients together, as in CyTOF), do the two C2S encodings keep **cell types** apart (cLISI)
and mix **patients** (iLISI)? This complements `ProfileVsPhysical` / `KnnEnrichmentIMC`, which work within one image.

**LISI** (Korsunsky et al. 2019; the `harmonypy.compute_lisi` algorithm, validated against harmonypy 0.0.10 below):
- 90 nearest neighbours (self excluded), Gaussian weights with a per-cell bandwidth calibrated to perplexity 30;
- LISI = 1 / Σ_c p_c², the effective number of label categories among a cell's neighbours.
- **cLISI** (labels = cell type): 1 = neighbours all one type. **Lower is better.**
- **iLISI** (labels = patient): higher = patients better mixed. **Higher is better.**

**Data and encodings:** identical to `ProfileVsPhysical` v3 / `ProfileVsPhysicalJackson`.
- Tumour images with at least 200 cells; Danenberg hot-aggregate cells removed.
- arcsinh(x / 1), per-image robust z clipped to ±5 (*intensity*); the within-cell rank of the clipped z, ties by column order (*rank*).
- Spaces: `cytof6`, the full panel (Danenberg 36, Jackson 32) and the same random 6/12/18/24/31-marker sets (same seed and code).
- Exact ties are broken with 1e-5 jitter, as in the cohort-pool analysis.

**Design.** LISI depends on the number of cells and categories, so each dataset is evaluated on identical draws:
- one draw = 10 random patients with at least 500 tumour cells, 500 random cells each (5,000 cells);
- labels: fine phenotype (Danenberg 32 / Jackson 27), broad class (epithelial / immune / stromal), patient (10);
- 100 draws for `cytof6` and the full panel, 20 draws for the random panels. Rank and intensity use the same cells in each draw (paired);
- Jackson only: 100 extra draws of 5 Basel + 5 Zurich patients, with iLISI over **cohort** (2 categories).

**Reference values** (per draw, from the same neighbours with the labels shuffled, i.e. perfect mixing):
- `iLISI_rel = (iLISI − 1) / (iLISI_shuffled − 1)`: 1 = patients fully mixed, 0 = each patient on its own;
- `cLISI_rel = (cLISI_shuffled − cLISI) / (cLISI_shuffled − 1)`: 1 = types fully separated, 0 = types random.

**Summary per draw** = median over cells (the scIB convention). Reported: the mean over draws with the 2.5–97.5 percentile range over draws,
and the paired rank − intensity difference.

**Caveat:** the phenotypes were defined by clustering intensity data, so cLISI favours intensity (the same caveat as T1 in `ProfileVsPhysical`).

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260927
OUT = "/kaggle/working"
COFACTOR = 1.0
MIN_CELLS = 200
PERPLEXITY = 30
N_PAT, N_PER = 10, 500
R_MAIN, R_SUB = 100, 20
SUBSET_SIZES = {6: 20, 12: 8, 18: 8, 24: 8, 31: 8}      # identical to ProfileVsPhysical v3
CYTOF6 = ["pan_cytokeratin", "cytokeratin_5", "cytokeratin_8_18", "sma", "estrogen_receptor_alpha", "ki_67"]
N_JOBS = os.cpu_count()

def find_in(kernel, name):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if kernel in h]
    assert len(hits) == 1, (kernel, name, hits)
    return hits[0]

## LISI (vectorised port of `harmonypy.compute_lisi`)

In [ ]:
def knn(V, k=PERPLEXITY * 3):
    """k nearest neighbours including self (column 0), as in harmonypy; self is dropped afterwards."""
    D = cdist(V, V)
    idx = np.argpartition(D, k - 1, axis=1)[:, :k]
    d = np.take_along_axis(D, idx, 1)
    o = np.argsort(d, axis=1, kind="stable")
    idx, d = np.take_along_axis(idx, o, 1), np.take_along_axis(d, o, 1)
    return d[:, 1:], idx[:, 1:]

def calibrate(D, perplexity=PERPLEXITY, tol=1e-5, n_tries=50):
    """Per-cell Gaussian weights with entropy log(perplexity): the binary search of harmonypy's compute_simpson."""
    n = len(D)
    logU = np.log(perplexity)
    beta, bmin, bmax = np.ones(n), np.full(n, -np.inf), np.full(n, np.inf)
    def hp(b):
        P = np.exp(-D * b[:, None])
        s = P.sum(1)
        with np.errstate(divide="ignore", invalid="ignore"):
            H = np.where(s > 0, np.log(s) + b * (D * P).sum(1) / s, 0.0)
            P = np.where(s[:, None] > 0, P / s[:, None], 0.0)
        return H, P
    H, P = hp(beta)
    for _ in range(n_tries):
        act = np.abs(H - logU) >= tol
        if not act.any():
            break
        pos, neg = act & (H - logU > 0), act & ~(H - logU > 0)
        bmin[pos] = beta[pos]
        beta[pos] = np.where(np.isfinite(bmax[pos]), (beta[pos] + bmax[pos]) / 2, beta[pos] * 2)
        bmax[neg] = beta[neg]
        beta[neg] = np.where(np.isfinite(bmin[neg]), (beta[neg] + bmin[neg]) / 2, beta[neg] / 2)
        Hn, Pn = hp(beta)
        H[act], P[act] = Hn[act], Pn[act]
    return H, P

def lisi_from(H, P, I, codes, n_cat):
    n, k = I.shape
    lab = codes[I]
    w = np.bincount((np.arange(n)[:, None] * n_cat + lab).ravel(), P.ravel(), n * n_cat).reshape(n, n_cat)
    simpson = (w ** 2).sum(1) + np.where(H == 0, -1.0, 0.0)
    return 1.0 / simpson

## Validation against harmonypy 0.0.10 (pure-Python reference) and sanity checks

In [ ]:
import subprocess, sys
ref_ok = None
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "harmonypy==0.0.10"], check=True, timeout=300)
    from harmonypy.lisi import compute_lisi as hp_lisi
    r = np.random.default_rng(1)
    Xs = np.r_[r.normal(0, 1, (400, 8)), r.normal(1.5, 1, (400, 8))]
    meta = pd.DataFrame({"a": np.r_[np.zeros(400, int), np.ones(400, int)].astype(str),
                         "b": r.integers(0, 5, 800).astype(str)})
    ref = hp_lisi(Xs, meta, ["a", "b"])
    D, I = knn(Xs)
    H, P = calibrate(D)
    mine = np.c_[[lisi_from(H, P, I, pd.factorize(meta[c])[0], meta[c].nunique()) for c in ["a", "b"]]].T
    ref_ok = float(np.abs(mine - ref).max())
    print("max |mine - harmonypy| =", ref_ok)
    assert ref_ok < 1e-3, "LISI port disagrees with harmonypy"
except (subprocess.SubprocessError, ImportError) as e:
    print("harmonypy not available (no internet?):", e)

r = np.random.default_rng(0)
Xs = np.r_[r.normal(0, 1, (500, 6)), r.normal(8, 1, (500, 6))]
D, I = knn(Xs); H, P = calibrate(D)
sep = np.median(lisi_from(H, P, I, np.r_[np.zeros(500, int), np.ones(500, int)], 2))
mix = np.median(lisi_from(H, P, I, r.integers(0, 2, 1000), 2))
print(f"separated labels: {sep:.3f} (expect ~1); random labels: {mix:.3f} (expect ~2)")
assert sep < 1.05 and mix > 1.8
print("sanity checks passed")

## Load both datasets (same filters and encodings as the earlier runs)

In [ ]:
def rank_vec(Zs):
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def load(name):
    kern = f"{name}-cells-prep"
    report = json.load(open(find_in(kern, "prep_report.json")))
    cells = pd.read_parquet(find_in(kern, "cells.parquet"))
    chan = list(report["channel_map"].values()) if name == "danenberg" else list(report["channel_map"])
    protein = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
    if name == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)].rename(columns={"metabric_id": "patient_id"})
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        keys = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
        broad = {p: "epithelial" if epi_frac[p] > 0.5 else "stromal" if any(s in p.lower() for s in keys) else "immune"
                 for p in epi_frac.index}
        cells["broad_class"] = cells.cell_phenotype.map(broad)
        cells["cohort"] = "danenberg"
    else:
        cells = cells[cells.is_tumour == 1]
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code = pd.factorize(cells.image_name, sort=True)[0]
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[protein].values.astype(np.float64) / COFACTOR)
    Z = np.empty_like(A, dtype=np.float32)
    for a, b in zip(starts[:-1], starts[1:]):
        med = np.median(A[a:b], 0)
        mad = np.median(np.abs(A[a:b] - med), 0)
        Z[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
    NP = len(protein)
    full = f"full{NP}"
    rng = np.random.default_rng(SEED)                   # same draws as ProfileVsPhysical v3 / ProfileVsPhysicalJackson
    col = {c: i for i, c in enumerate(protein)}
    spaces = {"cytof6": [col[c] for c in CYTOF6], full: list(range(NP))}
    seen = {tuple(sorted(spaces["cytof6"]))}
    for size, n_sub in SUBSET_SIZES.items():
        got = 0
        while got < n_sub:
            s = tuple(sorted(rng.choice(NP, size, replace=False).tolist()))
            if s not in seen:
                seen.add(s)
                spaces[f"rand{size}_{got:02d}"] = list(s)
                got += 1
    meta = cells[["patient_id", "cohort", "cell_phenotype", "broad_class"]].reset_index(drop=True)
    print(f"{name}: {len(cells):,} cells, {cells.image_name.nunique()} images, {cells.patient_id.nunique()} patients, {NP} markers")
    return dict(Z=Z, meta=meta, spaces=spaces, full=full, NP=NP, protein=protein)

DATA = {n: load(n) for n in ("danenberg", "jackson")}
json.dump({n: {s: [d["protein"][i] for i in v] for s, v in d["spaces"].items()} for n, d in DATA.items()},
          open(f"{OUT}/spaces.json", "w"), indent=1)

## Draws

In [ ]:
LABELS = ("cell_phenotype", "broad_class", "patient_id")

def size_of(space, NP):
    return NP if space.startswith("full") else 6 if space == "cytof6" else int(space[4:].split("_")[0])

def make_draws(meta, n_draws, rng, cohort_design=False):
    by_pat = meta.groupby("patient_id").indices
    cnt = pd.Series({p: len(v) for p, v in by_pat.items()})
    elig = np.array(sorted(cnt.index[cnt >= N_PER]))
    coh = meta.groupby("patient_id").cohort.first().reindex(elig).values
    draws = []
    for _ in range(n_draws):
        if cohort_design:
            pats = np.r_[rng.choice(elig[coh == "basel"], N_PAT // 2, replace=False),
                         rng.choice(elig[coh == "zurich"], N_PAT // 2, replace=False)]
        else:
            pats = rng.choice(elig, N_PAT, replace=False)
        draws.append(np.concatenate([rng.choice(by_pat[p], N_PER, replace=False) for p in pats]))
    return draws, len(elig)

def run_draw(dname, design, j, seed, spaces, NP, Zd, labvals):
    """Zd: the draw's cells x all protein markers; labvals: {label: values for the draw's cells}."""
    r = np.random.default_rng(seed)
    rows = []
    codes = {l: pd.factorize(v)[0] for l, v in labvals.items()}
    shuf = {l: r.permutation(c) for l, c in codes.items()}
    for sname, cols in spaces.items():
        Zs = Zd[:, cols]
        jit = r.normal(0, 1e-5, Zs.shape)
        for enc in ("intensity", "rank"):
            V = (Zs if enc == "intensity" else rank_vec(Zs)) + jit
            D, I = knn(V)
            H, P = calibrate(D)
            for l in codes:
                n_cat = int(codes[l].max()) + 1
                v = lisi_from(H, P, I, codes[l], n_cat)
                v0 = lisi_from(H, P, I, shuf[l], n_cat)
                rows.append(dict(dataset=dname, design=design, draw=j, space=sname, size=size_of(sname, NP), enc=enc,
                                 label=l, n_cat=n_cat, median=float(np.median(v)), mean=float(np.mean(v)),
                                 shuffled=float(np.median(v0))))
    return rows

t0 = time.time()
jobs = []
elig_n = {}
def job(dname, design, j, idx, seed, spaces, labels):
    """Only the draw's own rows are sent to the workers."""
    d = DATA[dname]
    return (dname, design, j, seed, spaces, d["NP"], d["Z"][idx], {l: d["meta"][l].values[idx] for l in labels})
for dname, d in DATA.items():
    rng = np.random.default_rng(SEED + (0 if dname == "danenberg" else 1))
    main = {s: d["spaces"][s] for s in ("cytof6", d["full"])}
    subs = {s: v for s, v in d["spaces"].items() if s.startswith("rand")}
    dr, elig_n[dname] = make_draws(d["meta"], R_MAIN, rng)
    jobs += [job(dname, "patients", j, idx, SEED + 1000 * j, main, LABELS) for j, idx in enumerate(dr)]
    jobs += [job(dname, "patients", j, idx, SEED + 1000 * j + 7, subs, LABELS) for j, idx in enumerate(dr[:R_SUB])]
    if dname == "jackson":
        dc, elig_n["jackson_cohort"] = make_draws(d["meta"], R_MAIN, rng, cohort_design=True)
        jobs += [job(dname, "cohorts", j, idx, SEED + 1000 * j + 3, main, LABELS + ("cohort",)) for j, idx in enumerate(dc)]
print("eligible patients (>= 500 tumour cells):", elig_n, "|", len(jobs), "jobs")
res = Parallel(n_jobs=N_JOBS, verbose=5)(delayed(run_draw)(*a) for a in jobs)
draws = pd.DataFrame([r for rows in res for r in rows])
draws.to_csv(f"{OUT}/lisi_draws.csv", index=False)
print(f"{len(draws):,} rows, {(time.time() - t0) / 60:.1f} min")

## Summaries

In [ ]:
draws["rel"] = np.where(draws.label.isin(["patient_id", "cohort"]),
                        (draws["median"] - 1) / (draws.shuffled - 1),
                        (draws.shuffled - draws["median"]) / (draws.shuffled - 1))
def pct(v):
    v = np.asarray(v, float)
    return v.mean(), np.percentile(v, 2.5), np.percentile(v, 97.5)

rows = []
main = draws[~draws.space.str.startswith("rand")].copy()
main["panel"] = np.where(main.space == "cytof6", "cytof6", "full")
for key, g in main.groupby(["dataset", "design", "panel", "label"]):
    row = dict(zip(["dataset", "design", "panel", "label"], key), n_cat=int(g.n_cat.iloc[0]), n_draws=g.draw.nunique())
    for enc in ("intensity", "rank"):
        e = g[g.enc == enc].sort_values("draw")
        for stat in ("median", "rel", "shuffled"):
            row[f"{enc}_{stat}"], row[f"{enc}_{stat}_lo"], row[f"{enc}_{stat}_hi"] = pct(e[stat].values)
    w = g.pivot_table(index="draw", columns="enc", values="median")
    row["diff"], row["diff_lo"], row["diff_hi"] = pct((w["rank"] - w["intensity"]).values)
    wr = g.pivot_table(index="draw", columns="enc", values="rel")
    row["diff_rel"], row["diff_rel_lo"], row["diff_rel_hi"] = pct((wr["rank"] - wr["intensity"]).values)
    rows.append(row)
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/lisi_summary.csv", index=False)
show = ["dataset", "design", "panel", "label", "n_cat", "intensity_median", "rank_median", "diff", "diff_lo", "diff_hi",
        "intensity_rel", "rank_rel", "intensity_shuffled"]
print(summary[show].round(3).to_string(index=False))

pat = draws.design == "patients"
cur = draws[pat & (draws.space.str.startswith("rand") | (draws.space.str.startswith("full") & (draws.draw < R_SUB)))]
cur = cur.groupby(["dataset", "size", "enc", "label", "draw"])[["median", "rel"]].mean().reset_index()
crow = []
for (dn, size, lab), g in cur.groupby(["dataset", "size", "label"]):
    w = g.pivot_table(index="draw", columns="enc", values="median")
    wr = g.pivot_table(index="draw", columns="enc", values="rel")
    d_, lo, hi = pct((w["rank"] - w["intensity"]).values)
    crow.append(dict(dataset=dn, size=size, label=lab, intensity=w["intensity"].mean(), rank=w["rank"].mean(),
                     diff=d_, diff_lo=lo, diff_hi=hi, intensity_rel=wr["intensity"].mean(), rank_rel=wr["rank"].mean()))
curve = pd.DataFrame(crow)
curve.to_csv(f"{OUT}/lisi_panel_curve.csv", index=False)
print(curve[curve.label != "broad_class"].round(3).to_string(index=False))

json.dump(dict(eligible_patients=elig_n, harmonypy_max_abs_diff=ref_ok, design=dict(N_PAT=N_PAT, N_PER=N_PER, R_MAIN=R_MAIN, R_SUB=R_SUB,
                                                                                    perplexity=PERPLEXITY),
               summary=summary.to_dict("records")), open(f"{OUT}/summary_lisi.json", "w"), indent=1, default=float)

## Figure: cLISI and iLISI vs panel size

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, dn in enumerate(("danenberg", "jackson")):
    for j, (lab, title) in enumerate((("cell_phenotype", "cLISI, fine phenotype (lower = types kept apart)"),
                                      ("patient_id", "iLISI, patient (higher = patients mixed)"))):
        ax = axes[i, j]
        g = curve[(curve.dataset == dn) & (curve.label == lab)].sort_values("size")
        for enc, c in (("intensity", "C0"), ("rank", "C1")):
            ax.plot(g["size"], g[enc], "o-", color=c, label=enc)
        ax.axvline(31, color="grey", ls=":", lw=0.8)
        ax.set_title(f"{dn}: {title}", fontsize=9); ax.set_xlabel("markers in panel"); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUT}/fig_lisi_panel_curve.png", dpi=150)
print("done")